# Imports

In [1]:
import torch
import torchvision
from torchvision.models import resnet18, densenet121
from torchvision.transforms import v2
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [3]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Attempt to locate the current directory of this notebook on your Google Drive
# In Colab, notebooks are typically located in '/content/drive/MyDrive/Colab Notebooks/' or a subfolder
# We will check if the default Colab path exists, or fallback to the root of your MyDrive
default_notebook_path = '/content/drive/MyDrive/Kristiania/5th_Semester/Deep_Learning/Mandatory_Assignment'
if os.path.exists(default_notebook_path):
    cifar10data = os.path.join(default_notebook_path, 'cifar10_data')
else:
    cifar10data = '/content/drive/MyDrive/Kristiania/5th_Semester/Deep_Learning/Mandatory_Assignment/cifar10_data'

os.makedirs(cifar10data, exist_ok=True)
print(f"CIFAR-10 datasets will be saved/loaded from: {cifar10data}")

Mounted at /content/drive
CIFAR-10 datasets will be saved/loaded from: /content/drive/MyDrive/Kristiania/5th_Semester/Deep_Learning/Mandatory_Assignment/cifar10_data


# Defining Three Architectures

In [4]:
# (I'll remove name later)
# Note for julien: Baseline, no change needed
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

# Note for julien: I'll change this one
def get_cifar_resnet18():
    model = resnet18(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)            # Replace the initial 7x7 convolution with a smaller 3x3 convolution
    model.maxpool = nn.Identity()                                                             # Remove the max pooling layer
    return model

# Note for julien: You change this one
def get_cifar_densenet121():
    model = densenet121(weights=None, num_classes=10)
    model.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)   # Modify the first convolutional layer and pooling for 32x32 inputs
    model.features.pool0 = nn.Identity()
    return model

architectures = {
    "baseline_simple_cnn": SimpleCNN,
    "resnet18": get_cifar_resnet18,
    "densenet121": get_cifar_densenet121
}

# Defining Three Augmentations

In [5]:
base_ops = [
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
]

augmentation_strategies = {
    # Note for julien: Baseline, no change needed
    "baseline_augmentation": v2.Compose(base_ops),

    # Note for julien: I'll change this one
    "aug1": v2.Compose([
        v2.ToImage(),
        v2.RandomCrop(32, padding=4),
        v2.RandomHorizontalFlip(),
        *base_ops
    ]),

    # Note for julien: You change this one
    "aug2": v2.Compose([
        v2.ToImage(),
        v2.RandomCrop(32, padding=4),
        v2.RandomHorizontalFlip(),
        v2.RandAugment(num_ops=2, magnitude=9),
        *base_ops
    ])
}

# Defining Three Optimizers

In [6]:
optimizers = {
    # Note for julien: Baseline, no change needed
    "baseline_sgd_momentum": lambda params: optim.SGD(params, lr=0.01, momentum=0.9),
    # Note for julien: I'll change this one
    "adam": lambda params: optim.Adam(params, lr=0.001),
    # Note for julien: You change this one
    "adamw": lambda params: optim.AdamW(params, lr=0.001, weight_decay=0.01)
}

# Selecting Configuration

In [7]:
train_transform = augmentation_strategies["baseline_augmentation"]
current_architecture = architectures["baseline_simple_cnn"]
current_optimizer = optimizers["baseline_sgd_momentum"]

# 1 Pre Processing

## 1.2 Seeding

In [8]:
experiment_seeds = [0, 1, 2]

def set_seed(seed):
    torch.manual_seed(seed)             # CPU Seed for Torch
    torch.cuda.manual_seed_all(seed)    # GPU Seed for Torch

current_seed = experiment_seeds[0]
set_seed(current_seed)

## 1.3 CIFAR-10 Dataset

### 1.3.1 Downloading and Tranforming

In [9]:
'''transform = v2.Compose([
    v2.ToImage(),                                     # Transforms the PIL Image into a torchvision.tv_tensors._image.Image (aka Tensor)
    # Augmentations step here
    v2.ToDtype(torch.float32, scale=True),            # Changes Dtype and scales values down to [0, 1]
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])  # Normalize values with mean and std of 0.5, giving us the range [-1, 1]

batch_size = 128

trainset = torchvision.datasets.CIFAR10(root=cifar10data, train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root=cifar10data, train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')'''

"transform = v2.Compose([\n    v2.ToImage(),                                     # Transforms the PIL Image into a torchvision.tv_tensors._image.Image (aka Tensor)\n    # Augmentations step here\n    v2.ToDtype(torch.float32, scale=True),            # Changes Dtype and scales values down to [0, 1]\n    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])  # Normalize values with mean and std of 0.5, giving us the range [-1, 1]\n\nbatch_size = 128\n\ntrainset = torchvision.datasets.CIFAR10(root=cifar10data, train=True, download=True, transform=transform)\ntrainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)\n\ntestset = torchvision.datasets.CIFAR10(root=cifar10data, train=False, download=True, transform=transform)\ntestloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)\n\nclasses = ('plane', 'car', 'bird', 'cat',\n           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')"

In [11]:
# Assign the transforms to the training set
train_transform = train_transform


# Transform for validation and test data (NO augmentations)
eval_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

full_trainset_aug = torchvision.datasets.CIFAR10(root=cifar10data, train=True, download=True, transform=train_transform)
full_trainset_eval = torchvision.datasets.CIFAR10(root=cifar10data, train=True, download=True, transform=eval_transform)

split_generator = torch.Generator().manual_seed(42)                 # Hard coding Seeding for generator (Should be the same every time)
indices = torch.randperm(50000, generator=split_generator).tolist()
train_idx, val_idx = indices[:45000], indices[45000:]

trainset = torch.utils.data.Subset(full_trainset_aug, train_idx)    # map the training indices to the augmented data
valset = torch.utils.data.Subset(full_trainset_eval, val_idx)       # map the valuation indices to the normalized data

# The test set only uses the evaluation transform
testset = torchvision.datasets.CIFAR10(root=cifar10data, train=False, download=True, transform=eval_transform)

In [12]:
batch_size = 128
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

### 1.3.2 Step by Step Inspection for understanding underlying data

In [13]:
pretransform = torchvision.datasets.CIFAR10(root=cifar10data, train=True, download=True, transform=None)

In [14]:
print("--Before Transformation Shape and Dtype--\n ---------------------------------------")
pimg, plabel = pretransform[0]
print(type(pimg))
print(f"Size (W, H): {pimg.size}")
print(f"Mode: {pimg.mode}")

print("\n--After Transformation Shape and Dtype--\n ---------------------------------------")
img, label = trainset[0]
print(type(img))
print(img.shape)
print(img.dtype)

step1_to_image = v2.ToImage()
step2_to_dtype = v2.ToDtype(torch.float32, scale=True)
step3_normalize = v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

img_t1 = step1_to_image(pimg)
img_t2 = step2_to_dtype(img_t1)
img_t3 = step3_normalize(img_t2)

print("\n-- Step-by-Step --")
print(f"After ToImage:   type={type(img_t1)}, shape={img_t1.shape}, dtype={img_t1.dtype}, min={img_t1.min()}, max={img_t1.max()}")
print(f"After ToDtype:   type={type(img_t2)}, shape={img_t2.shape}, dtype={img_t2.dtype}, min={img_t2.min():.2f}, max={img_t2.max():.2f}")
print(f"After Normalize: type={type(img_t3)}, shape={img_t3.shape}, dtype={img_t3.dtype}, min={img_t3.min():.2f}, max={img_t3.max():.2f}")

--Before Transformation Shape and Dtype--
 ---------------------------------------
<class 'PIL.Image.Image'>
Size (W, H): (32, 32)
Mode: RGB

--After Transformation Shape and Dtype--
 ---------------------------------------
<class 'torchvision.tv_tensors._image.Image'>
torch.Size([3, 32, 32])
torch.float32

-- Step-by-Step --
After ToImage:   type=<class 'torchvision.tv_tensors._image.Image'>, shape=torch.Size([3, 32, 32]), dtype=torch.uint8, min=0, max=255
After ToDtype:   type=<class 'torchvision.tv_tensors._image.Image'>, shape=torch.Size([3, 32, 32]), dtype=torch.float32, min=0.00, max=1.00
After Normalize: type=<class 'torchvision.tv_tensors._image.Image'>, shape=torch.Size([3, 32, 32]), dtype=torch.float32, min=-1.00, max=1.00


In [15]:
# 1. Hardware setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. Instantiate the model and move it to the GPU/CPU
model = current_architecture()
model = model.to(device)

# 4. Instantiate the optimiser by passing the initialized model parameters
optimiser = current_optimizer(model.parameters())

# 5. Define the loss function
criterion = nn.CrossEntropyLoss()

# --- THE TRAINING LOOP ---

# Set model to training mode (enables layers like Dropout and BatchNorm)
model.train()

epochs = 30
for epoch in range(epochs):
    running_loss = 0.0

    # Iterate over the DataLoader (which already applies the chosen augmentation)
    for batch_idx, (inputs, labels) in enumerate(trainloader):

        # Move data to the active device
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Step 1: Zero the gradients
        # PyTorch accumulates gradients, so they must be cleared before every batch
        optimiser.zero_grad()

        # Step 2: Forward propagation
        # Pass the images through the network to get the predictions
        outputs = model(inputs)

        # Step 3: Compute the loss
        # Calculate how far off the predictions were from the actual labels
        loss = criterion(outputs, labels)

        # Step 4: Backward propagation
        # Compute the gradient of the loss with respect to all model weights
        loss.backward()

        # Step 5: Optimiser step
        # Adjust the model weights based on the computed gradients
        optimiser.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1} complete. Average Loss: {running_loss / len(trainloader):.4f}")

Epoch 1 complete. Average Loss: 1.4371
Epoch 2 complete. Average Loss: 1.0578
Epoch 3 complete. Average Loss: 0.8441
Epoch 4 complete. Average Loss: 0.6692
Epoch 5 complete. Average Loss: 0.5016
Epoch 6 complete. Average Loss: 0.4396
Epoch 7 complete. Average Loss: 0.3649
Epoch 8 complete. Average Loss: 0.3569
Epoch 9 complete. Average Loss: 0.3423
Epoch 10 complete. Average Loss: 0.3604
Epoch 11 complete. Average Loss: 0.3850
Epoch 12 complete. Average Loss: 0.4538
Epoch 13 complete. Average Loss: 0.4469
Epoch 14 complete. Average Loss: 0.4617
Epoch 15 complete. Average Loss: 0.5116
Epoch 16 complete. Average Loss: 0.5009
Epoch 17 complete. Average Loss: 0.6614
Epoch 18 complete. Average Loss: 0.5762
Epoch 19 complete. Average Loss: 0.5714
Epoch 20 complete. Average Loss: 0.7742
Epoch 21 complete. Average Loss: 0.7770


KeyboardInterrupt: 